# Добавить БД 2026 недостающие климатические данные в из БД 2023

In [1]:
from os import getenv
from datetime import date

import pandas as pd
from sqlalchemy import create_engine, text

In [2]:
engine2026 = create_engine(getenv("SQLALCHEMY_RELOHELPER_URL"))
engine2023 = create_engine(getenv("SQLALCHEMY_RELOHELPER_URL_2023"))

In [6]:
# Загрузка данных из main df
df_pkl = pd.read_pickle("./data/geonameid.pkl")

# Загрузка данных из Relohelper2026
df2026 = pd.read_sql(
    "SELECT * FROM avg_climate", engine2026
)

# Загрузка данных из Relohelper2023
df2023 = pd.read_sql("SELECT * FROM avg_climate", engine2023)

In [8]:
df_pkl.head(3)

,city,state_code,state_name,country_code,country,city_country,cost_of_living,rent,cost_of_living_plus_rent,groceries,...,pollution,link,country_code2,geonameid,population,latitude,longitude,timezone,match_count,match_source
city_id,,,,,,,,,,,,,,,,,,,,,
3,Zurich,NaN,NaN,CHE,Switzerland,"Zurich, Switzerland",123.9,71.3,99.6,125.6,...,25.3,https://www.numbeo.com/cost-of-living/in/Zuric...,CH,2657896.0,415367.0,47.36667,8.55000,Europe/Zurich,1,alt_exact:Zurich
8,Geneva,NaN,NaN,CHE,Switzerland,"Geneva, Switzerland",118.4,64.3,93.4,116.0,...,24.2,https://www.numbeo.com/cost-of-living/in/Genev...,CH,2660646.0,201741.0,46.20222,6.14569,Europe/Zurich,1,alt_exact:Geneva
2,Basel,NaN,NaN,CHE,Switzerland,"Basel, Switzerland",114.0,47.3,83.2,113.7,...,31.4,https://www.numbeo.com/cost-of-living/in/Basel...,CH,2661604.0,177595.0,47.55839,7.57327,Europe/Zurich,1,name_exact:Basel


In [9]:
df2026.head(3)

,city_id,geonameid,month,high_temp,low_temp,pressure,wind_speed,humidity,rainfall,rainfall_days,...,snowfall_days,sea_temp,daylight,sunshine,sunshine_days,uv_index,cloud_cover,visibility,updated_date,updated_by
0,345,3183875,1,9.2,3.4,1017.2,10.6,74.0,86.0,14.1,...,0.4,NaN,9.7,6.0,16.0,3.0,36.0,10.0,2026-03-13,de_k2
1,345,3183875,2,10.7,4.3,1015.4,11.0,75.0,81.0,14.2,...,0.7,NaN,10.7,6.0,12.8,3.0,37.0,10.0,2026-03-13,de_k2
2,345,3183875,3,13.8,6.5,1015.1,10.2,71.0,79.0,15.3,...,0.0,NaN,12.0,7.8,14.1,4.0,33.0,10.0,2026-03-13,de_k2


In [10]:
df2023.head(3)

,city_id,month,high_temp,low_temp,pressure,wind_speed,humidity,rainfall,rainfall_days,snowfall,snowfall_days,sea_temp,daylight,sunshine,sunshine_days,uv_index,cloud_cover,visibility,sys_updated_date,sys_updeted_by
0,345,1,9.2,3.4,1017.2,10.6,74.0,86.0,14.1,1.0,0.4,NaN,9.0,6.0,16.0,3.0,36.0,10.0,2023-08-09,de_k2
1,345,2,10.7,4.3,1015.4,11.0,75.0,81.0,14.2,10.0,0.7,NaN,10.0,6.0,12.8,3.0,37.0,10.0,2023-08-09,de_k2
2,345,3,13.8,6.5,1015.1,10.2,71.0,79.0,15.3,0.0,0.0,NaN,12.0,7.0,14.1,4.0,33.0,10.0,2023-08-09,de_k2


анализ:

In [12]:
# 1) какие city_id есть в df_pkl, но нет в df2026
# missing_in_2026 = set(df_pkl["city_id"]) - set(df2026["city_id"])
# len(missing_in_2026), list(missing_in_2026)[:20]

# если в df_pkl city_id — индекс, то возьмём его как set(df_pkl.index)
missing_in_2026 = set(df_pkl.index) - set(df2026["city_id"])
len(missing_in_2026), list(missing_in_2026)[:20]

(45,
 [518,
  524,
  561,
  435,
  564,
  565,
  567,
  572,
  576,
  450,
  580,
  329,
  586,
  587,
  461,
  471,
  472,
  600,
  602,
  475])

In [13]:
# 2) из этих недостающих, какие есть в df2023
present_in_2023 = missing_in_2026 & set(df2023["city_id"])
len(present_in_2023), list(present_in_2023)[:20]

(12, [450, 518, 491, 524, 461, 495, 435, 471, 472, 506, 475, 479])

In [14]:
# городa, которые есть в df_pkl и в df2023, но отсутствуют в df2026
to_add_ids = sorted(present_in_2023)

# пустой фрейм с колонками как у df2026
df2026_add = pd.DataFrame(columns=df2026.columns)

# заполняем city_id + geonameid из df_pkl (индекс = city_id)
df2026_add["city_id"] = to_add_ids
df2026_add["geonameid"] = df_pkl.loc[to_add_ids, "geonameid"].values

In [16]:
# остальные колонки останутся NaN
df2026_add

,city_id,geonameid,month,high_temp,low_temp,pressure,wind_speed,humidity,rainfall,rainfall_days,...,snowfall_days,sea_temp,daylight,sunshine,sunshine_days,uv_index,cloud_cover,visibility,updated_date,updated_by
0,435,1566083.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,450,1581130.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,461,703448.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,471,1583992.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,472,709930.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,475,698740.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,479,706483.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,491,3435910.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,495,702550.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,506,1283240.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [18]:
df2026_add = df2023[df2023["city_id"].isin(to_add_ids)].copy().reset_index(drop=True)
df2026_add["geonameid"] = df2026_add["city_id"].map(df_pkl["geonameid"])

In [19]:
df2026_add

,city_id,month,high_temp,low_temp,pressure,wind_speed,humidity,rainfall,rainfall_days,snowfall,...,sea_temp,daylight,sunshine,sunshine_days,uv_index,cloud_cover,visibility,sys_updated_date,sys_updeted_by,geonameid
0,491,1,30.9,20.0,1011.8,13.9,56.0,42.0,7.5,NaN,...,25.0,14.0,10.0,21.6,7.0,17.0,10.0,2023-08-09,de_k2,3435910.0
1,491,2,29.5,19.9,1012.4,13.4,61.0,59.0,9.8,NaN,...,23.7,13.0,9.0,17.9,6.0,20.0,10.0,2023-08-09,de_k2,3435910.0
2,491,3,26.2,17.2,1014.7,13.2,65.0,49.0,9.6,NaN,...,22.8,12.0,8.0,20.3,6.0,22.0,10.0,2023-08-09,de_k2,3435910.0
3,491,4,22.8,14.5,1016.6,13.6,69.0,44.0,9.5,NaN,...,19.2,11.0,8.0,19.8,5.0,27.0,10.0,2023-08-09,de_k2,3435910.0
4,491,5,18.7,12.2,1017.9,13.6,76.0,46.0,8.5,NaN,...,16.1,10.0,6.0,19.8,4.0,35.0,10.0,2023-08-09,de_k2,3435910.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
139,472,8,29.3,17.3,1014.4,12.6,48.0,19.0,5.2,0.0,...,NaN,14.0,11.0,24.5,6.0,21.0,10.0,2023-08-09,de_k2,709930.0
140,472,9,22.5,12.4,1016.3,13.1,56.0,21.0,7.5,0.0,...,NaN,12.0,9.0,21.5,4.0,29.0,10.0,2023-08-09,de_k2,709930.0
141,472,10,13.9,6.5,1020.2,13.5,67.0,22.0,7.1,3.0,...,NaN,10.0,6.0,22.3,2.0,35.0,10.0,2023-08-09,de_k2,709930.0
142,472,11,7.0,1.9,1021.2,14.8,78.0,20.0,6.0,19.0,...,NaN,9.0,4.0,20.6,2.0,49.0,9.0,2023-08-09,de_k2,709930.0


In [20]:
# переименуем нужные колонки
df2026_add = df2026_add.rename(
    columns={
        "sys_updated_date": "updated_date",
        "sys_updeted_by": "updeted_by",
    }
)

# привести порядок колонок к такому же, как в df2026
df2026_add = df2026_add.reindex(columns=df2026.columns)

# проверка: имена + порядок колонок совпадают
assert list(df2026_add.columns) == list(df2026.columns)

In [22]:
df2026_add

,city_id,geonameid,month,high_temp,low_temp,pressure,wind_speed,humidity,rainfall,rainfall_days,...,snowfall_days,sea_temp,daylight,sunshine,sunshine_days,uv_index,cloud_cover,visibility,updated_date,updated_by
0,491,3435910.0,1,30.9,20.0,1011.8,13.9,56.0,42.0,7.5,...,NaN,25.0,14.0,10.0,21.6,7.0,17.0,10.0,2023-08-09,NaN
1,491,3435910.0,2,29.5,19.9,1012.4,13.4,61.0,59.0,9.8,...,NaN,23.7,13.0,9.0,17.9,6.0,20.0,10.0,2023-08-09,NaN
2,491,3435910.0,3,26.2,17.2,1014.7,13.2,65.0,49.0,9.6,...,NaN,22.8,12.0,8.0,20.3,6.0,22.0,10.0,2023-08-09,NaN
3,491,3435910.0,4,22.8,14.5,1016.6,13.6,69.0,44.0,9.5,...,NaN,19.2,11.0,8.0,19.8,5.0,27.0,10.0,2023-08-09,NaN
4,491,3435910.0,5,18.7,12.2,1017.9,13.6,76.0,46.0,8.5,...,NaN,16.1,10.0,6.0,19.8,4.0,35.0,10.0,2023-08-09,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
139,472,709930.0,8,29.3,17.3,1014.4,12.6,48.0,19.0,5.2,...,0.0,NaN,14.0,11.0,24.5,6.0,21.0,10.0,2023-08-09,NaN
140,472,709930.0,9,22.5,12.4,1016.3,13.1,56.0,21.0,7.5,...,0.0,NaN,12.0,9.0,21.5,4.0,29.0,10.0,2023-08-09,NaN
141,472,709930.0,10,13.9,6.5,1020.2,13.5,67.0,22.0,7.1,...,0.3,NaN,10.0,6.0,22.3,2.0,35.0,10.0,2023-08-09,NaN
142,472,709930.0,11,7.0,1.9,1021.2,14.8,78.0,20.0,6.0,...,2.6,NaN,9.0,4.0,20.6,2.0,49.0,9.0,2023-08-09,NaN


In [23]:
# из-за опечатки в df2023 "sys_updeted_by"...
df2026_add["updated_by"] = "de_k2"

In [24]:
df2026_add

,city_id,geonameid,month,high_temp,low_temp,pressure,wind_speed,humidity,rainfall,rainfall_days,...,snowfall_days,sea_temp,daylight,sunshine,sunshine_days,uv_index,cloud_cover,visibility,updated_date,updated_by
0,491,3435910.0,1,30.9,20.0,1011.8,13.9,56.0,42.0,7.5,...,NaN,25.0,14.0,10.0,21.6,7.0,17.0,10.0,2023-08-09,de_k2
1,491,3435910.0,2,29.5,19.9,1012.4,13.4,61.0,59.0,9.8,...,NaN,23.7,13.0,9.0,17.9,6.0,20.0,10.0,2023-08-09,de_k2
2,491,3435910.0,3,26.2,17.2,1014.7,13.2,65.0,49.0,9.6,...,NaN,22.8,12.0,8.0,20.3,6.0,22.0,10.0,2023-08-09,de_k2
3,491,3435910.0,4,22.8,14.5,1016.6,13.6,69.0,44.0,9.5,...,NaN,19.2,11.0,8.0,19.8,5.0,27.0,10.0,2023-08-09,de_k2
4,491,3435910.0,5,18.7,12.2,1017.9,13.6,76.0,46.0,8.5,...,NaN,16.1,10.0,6.0,19.8,4.0,35.0,10.0,2023-08-09,de_k2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
139,472,709930.0,8,29.3,17.3,1014.4,12.6,48.0,19.0,5.2,...,0.0,NaN,14.0,11.0,24.5,6.0,21.0,10.0,2023-08-09,de_k2
140,472,709930.0,9,22.5,12.4,1016.3,13.1,56.0,21.0,7.5,...,0.0,NaN,12.0,9.0,21.5,4.0,29.0,10.0,2023-08-09,de_k2
141,472,709930.0,10,13.9,6.5,1020.2,13.5,67.0,22.0,7.1,...,0.3,NaN,10.0,6.0,22.3,2.0,35.0,10.0,2023-08-09,de_k2
142,472,709930.0,11,7.0,1.9,1021.2,14.8,78.0,20.0,6.0,...,2.6,NaN,9.0,4.0,20.6,2.0,49.0,9.0,2023-08-09,de_k2


In [25]:
# дописать строки из df2026_add в таблицу avg_climate (append), без индекса
df2026_add.to_sql("avg_climate", engine2026, if_exists="append", index=False)

144